# 05 — RAG Prototyping

Purpose: build and inspect the retrieval side of RAG for one sample
ticker -- chunking, embeddings, FAISS search quality -- before trusting
`src/rag/build_index.py` and `retriever.py` across all 150 tickers.
Groq generation is tested separately in notebook 06 so retrieval quality
can be judged on its own first.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import yaml
import pandas as pd
from src.utils.s3_io import read_parquet_s3, list_keys
from src.rag.build_index import build_ticker_corpus, _chunk_text, _get_embedder

with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

bucket = config["s3"]["bucket"]
rag_config = config["rag"]


/home/moksha/EquiRisk/venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## 1. Pick a sample ticker and load its feature rows


In [9]:
from src.utils.s3_io import read_hive_partitioned_parquet_s3

prefix = config["s3"]["paths"]["processed_features"]
full_df = read_hive_partitioned_parquet_s3(prefix, bucket, partition_col="ticker")

sample_ticker = full_df["ticker"].unique()[0]
ticker_df = full_df[full_df["ticker"] == sample_ticker]
print(sample_ticker, ticker_df.shape)
ticker_df.tail(3)

360ONE (1238, 25)


,date,open,high,low,close,volume,headlines,descriptions,article_count,prev_close,...,ma_60d,ma_90d,rsi_14,macd_line,macd_signal,forward_volatility,risk_label,daily_sentiment,sentiment_3d_avg,ticker
1235,2026-07-22,1114.000000,1128.000000,1087.800049,1093.599976,913719,None,None,0,1119.699951,...,1098.778339,1071.020645,54.720639,1.853831,-0.772088,0.019193,Medium,0.0,0.0,360ONE
1236,2026-07-23,1080.000000,1098.900024,1072.300049,1083.199951,1503658,None,None,0,1093.599976,...,1099.587504,1071.706529,45.743838,-1.655784,-0.622659,NaN,None,0.0,0.0,360ONE
1237,2026-07-24,1068.300049,1105.000000,1065.000000,1102.300049,918262,None,None,0,1083.199951,...,1100.715004,1072.654348,44.881970,0.727558,-0.321804,NaN,None,0.0,0.0,360ONE


## 2. Build the raw text corpus for this ticker

In [10]:
docs = build_ticker_corpus(ticker_df, sample_ticker)
print(f"{len(docs)} raw documents")
for d in docs[:5]:
    print("-", d)

1 raw documents
- As of 2026-07-24, 360ONE has a risk label of None. 20-day volatility is 0.0209, 60-day volatility is 0.0175. Recent average news sentiment (3-day) is 0.000 (range -1 very negative to +1 very positive), based on 0 recent articles.


## 3. Chunk the corpus

In [11]:
all_chunks = []
for doc in docs:
    all_chunks.extend(_chunk_text(doc, rag_config["chunk_size_tokens"], rag_config["chunk_overlap_tokens"]))

print(f"{len(all_chunks)} chunks total")
for c in all_chunks[:5]:
    print("-", c)

1 chunks total
- As of 2026-07-24, 360ONE has a risk label of None. 20-day volatility is 0.0209, 60-day volatility is 0.0175. Recent average news sentiment (3-day) is 0.000 (range -1 very negative to +1 very positive), based on 0 recent articles.


## 4. Embed and build a FAISS index in-memory (no S3 write yet)

In [12]:
import faiss
import numpy as np

embedder = _get_embedder(rag_config["embedding_model"])
embeddings = embedder.encode(all_chunks, convert_to_numpy=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))
print("Index size:", index.ntotal, " Embedding dim:", embeddings.shape[1])


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Index size: 1  Embedding dim: 384


## 5. Try a few test queries and eyeball retrieval quality

In [13]:
test_queries = [
    "why is this stock risky right now",
    "what has recent news said about this company",
    "how volatile has this stock been lately",
]

for q in test_queries:
    q_vec = embedder.encode([q], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_vec, min(5, len(all_chunks)))
    print(f"\nQuery: {q}")
    for score, i in zip(scores[0], idx[0]):
        if 0 <= i < len(all_chunks):
            print(f"  [{score:.3f}] {all_chunks[i]}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: why is this stock risky right now
  [0.371] As of 2026-07-24, 360ONE has a risk label of None. 20-day volatility is 0.0209, 60-day volatility is 0.0175. Recent average news sentiment (3-day) is 0.000 (range -1 very negative to +1 very positive), based on 0 recent articles.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: what has recent news said about this company
  [0.257] As of 2026-07-24, 360ONE has a risk label of None. 20-day volatility is 0.0209, 60-day volatility is 0.0175. Recent average news sentiment (3-day) is 0.000 (range -1 very negative to +1 very positive), based on 0 recent articles.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: how volatile has this stock been lately
  [0.441] As of 2026-07-24, 360ONE has a risk label of None. 20-day volatility is 0.0209, 60-day volatility is 0.0175. Recent average news sentiment (3-day) is 0.000 (range -1 very negative to +1 very positive), based on 0 recent articles.


If retrieved chunks look off-topic or low-relevance, things worth
tuning in `config.yaml`'s `rag:` section: `chunk_size_tokens`,
`top_k_retrieval`, or the `embedding_model` itself.


## 6. Build the real index for this ticker via the production function (writes to S3)

In [14]:
# from src.rag.build_index import build_index_for_ticker
# build_index_for_ticker(sample_ticker, ticker_df, config)
